<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/pytorch-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="text-align: right; flex: 1;">
        <a href="https://pytorch.org/" target="_blank" rel="noopener noreferrer">
            <img src="../../../assets/images/pytorch/logo/pytorch-logo-dark.svg" 
                 alt="PyTorch Logo"
                 style="max-height: 48px; width: auto; background-color: #ffffff; border-radius: 8px;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependensies](#toc1_)    
- [LeNet-5](#toc2_)    
  - [Manual Implementation](#toc2_1_)    
    - [Initialize the Model](#toc2_1_1_)    
    - [Model Summary](#toc2_1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependensies](#toc0_)

📝 **Docs**:

- **Models and pre-trained weights:** [docs.pytorch.org/vision/stable/models.html](https://docs.pytorch.org/vision/stable/models.html)


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torchinfo import summary

# <a id='toc2_'></a>[LeNet-5](#toc0_)

- One of the pioneering **Convolutional Neural Network (CNN)** architectures developed in 1998 by [Yann LeCun](https://en.wikipedia.org/wiki/Yann_LeCun) and his colleagues at [AT&T Bell Labs](https://en.wikipedia.org/wiki/Bell_Labs)
- It was introduced in the landmark paper *[Gradient-based learning applied to document recognition](https://ieeexplore.ieee.org/document/726791)*, which demonstrated end-to-end learning directly from raw pixel data
- It was designed primarily for **handwritten digit recognition**, such as ZIP code and bank check processing
- It was trained on the *[MNIST dataset](http://yann.lecun.com/exdb/mnist/)*, where the original **28×28 images were padded to 32×32** to match the network architecture [[MNIST viewer](https://observablehq.com/@davidalber/mnist-browser)]
- LeNet-5 demonstrated that CNNs can automatically learn **spatial hierarchies of features**, eliminating the need for manual feature engineering and establishing the foundation for modern computer vision systems

<div style="text-align: center; padding-top: 10px;">
    <img src="../../../assets/images/original/cnn/architectures/lenet5.svg" alt="lenet5-architecture.svg" style="min-width: 256px; max-width: 100%; height: auto; border-radius: 16px;">
    <p><em>Figure 1: LeNet-5 Architecture</em></p>
</div>


## <a id='toc2_1_'></a>[Manual Implementation](#toc0_)


### <a id='toc2_1_1_'></a>[Initialize the Model](#toc0_)


In [ ]:
# learnable subsampling layer (original LeNet-5 S2, S4)
class Subsampling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.scale = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))

    def forward(self, x):
        x = self.pool(x)
        x = self.scale.view(1, -1, 1, 1) * x + self.bias.view(1, -1, 1, 1)
        return x

In [ ]:
# gaussian RBF output layer (original LeNet-5)
class RBFLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(out_features, in_features))
        self.sigma = nn.Parameter(torch.ones(out_features))

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, features)
        centers = self.centers.unsqueeze(0)  # (1, classes, features)
        dist = torch.sum((x - centers) ** 2, dim=2)
        return torch.exp(-dist / (2 * self.sigma**2))

In [ ]:
class LeNet5(nn.Module):

    def __init__(self):
        super().__init__()

        # C1
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)

        # S2
        self.sub2 = Subsampling(6)

        # C3 (full connection approximation)
        self.conv3 = nn.Conv2d(6, 16, kernel_size=5)

        # S4
        self.sub4 = Subsampling(16)

        # C5
        self.conv5 = nn.Conv2d(16, 120, kernel_size=5)

        # F6
        self.fc6 = nn.Linear(120, 84)

        # output (RBF)
        self.rbf = RBFLayer(84, 10)

    def forward(self, x):

        # C1
        x = torch.tanh(self.conv1(x))

        # S2
        x = torch.tanh(self.sub2(x))

        # C3
        x = torch.tanh(self.conv3(x))

        # S4
        x = torch.tanh(self.sub4(x))

        # C5
        x = torch.tanh(self.conv5(x))

        # flatten
        x = x.view(x.size(0), -1)

        # F6
        x = torch.tanh(self.fc6(x))

        # output
        x = self.rbf(x)

        return x

In [ ]:
# initialize a model
lenet5_manual = LeNet5()
lenet5_manual

### <a id='toc2_1_2_'></a>[Model Summary](#toc0_)


In [ ]:
# model summary report
summary(lenet5_manual, input_size=(1, 1, 32, 32), device="cpu")